In [1]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Find all CSV files
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Data directory: {DATA_DIR}")
print(f"Total CSV files: {len(csv_files)}")
print("-" * 60)

for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:.2f} MB")


Data directory: c:\Users\aibib\OneDrive\python-workspace\freedomprofile\data\raw
Total CSV files: 5
------------------------------------------------------------
SAPP Покупки у партнеров.csv: 16.75 MB
SAPP пользователи.csv: 44.72 MB
SAPP привлечения.csv: 18.63 MB
SAPP Процессы пользователей в приложении.csv: 858.78 MB
SAPP транзакции.csv: 540.37 MB


In [2]:
# Inspect the structure of each CSV file

for file in csv_files:
    print(f"\n{'=' * 70}")
    print(f"FILE: {file.name}")
    print(f"{'=' * 70}")

    # Read only the first 5 rows
    df_sample = pd.read_csv(
        file,
        nrows=5,
        low_memory=False
    )

    # Display column names and data types
    print("\nCOLUMNS AND DATA TYPES:")
    print(df_sample.dtypes)

    # Display sample records
    print("\nSAMPLE DATA:")
    display(df_sample)

    print(f"\nTotal columns: {len(df_sample.columns)}")



FILE: SAPP Покупки у партнеров.csv

COLUMNS AND DATA TYPES:
purchase_date       object
remote_flag         object
customer_id        float64
app_name            object
purchase_amount    float64
counter              int64
cashback_amount    float64
dtype: object

SAMPLE DATA:


,purchase_date,remote_flag,customer_id,app_name,purchase_amount,counter,cashback_amount
0,2026-03-03,Оплата в Superapp,4008474.0,groceries,-65736.0,1,-94121.49
1,2026-05-10,Оплата в Superapp,4155630.0,ticketon,-88167.0,1,-99507.00
2,2026-05-06,Оплата в Superapp,3966618.0,ticketon,-83667.0,1,-98787.00
3,2026-04-07,Оплата в Superapp,4084518.0,train_tickets,-92448.0,1,-100652.91
4,2026-01-05,Оплата в Superapp,3757106.0,Тикетон,-95067.0,1,-100479.00



Total columns: 7

FILE: SAPP пользователи.csv

COLUMNS AND DATA TYPES:
city             object
customer_id     float64
gender           object
customer_age      int64
reg_date         object
dtype: object

SAMPLE DATA:


,city,customer_id,gender,customer_age,reg_date
0,Актау,1481514.0,F,31,2026-02-23
1,Талдыкорган,3896250.0,M,39,2026-01-16
2,Кызылорда,3963568.0,F,79,2026-01-24
3,Шымкент,4005274.0,F,48,2026-01-29
4,Алматы,4053200.0,M,58,2026-02-05



Total columns: 5

FILE: SAPP привлечения.csv

COLUMNS AND DATA TYPES:
secondary_category_filled     object
customer_id                  float64
dtype: object

SAMPLE DATA:


,secondary_category_filled,customer_id
0,bank,242220.0
1,Ticketon,248836.0
2,NaN,263180.0
3,NaN,264776.0
4,Ticketon,269296.0



Total columns: 2

FILE: SAPP Процессы пользователей в приложении.csv

COLUMNS AND DATA TYPES:
customer_id     float64
process_code     object
lang             object
status           object
started_at       object
completed_at     object
dtype: object

SAMPLE DATA:


,customer_id,process_code,lang,status,started_at,completed_at
0,971302.0,LivenessProcess,RU,ERROR,2025-01-09 13:56:17.737,2025-01-09 13:57:38.282
1,970874.0,CustomerRegistrationProcess,RU,DECLINED,2025-01-09 14:04:06.858,2025-01-09 14:08:58.462
2,965574.0,CustomerRegistrationProcess,RU,ERROR,2025-01-08 21:42:00.916,2025-01-08 21:53:34.064
3,1002080.0,LivenessProcess,RU,COMPLETED,2025-01-10 18:49:17.196,2025-01-10 18:50:14.698
4,963542.0,LivenessProcess,RU,DECLINED,2025-01-08 19:32:32.030,2025-01-08 19:34:43.809



Total columns: 6

FILE: SAPP транзакции.csv

COLUMNS AND DATA TYPES:
transaction_date       object
transaction_id          int64
operation_type         object
transaction_sum       float64
transaction_status     object
mcc                     int64
terminal_type          object
customer_id           float64
dtype: object

SAMPLE DATA:


,transaction_date,transaction_id,operation_type,transaction_sum,transaction_status,mcc,terminal_type,customer_id
0,2026-01-01,2601010547384688,P2P Credit,-101517.00,Успешные,6536,ePOS,499782.0
1,2026-01-01,2601010547475905,Purchase,-101623.77,AUSR0401,5814,POS,315852.0
2,2026-01-01,2601010547483150,Purchase,-97946.58,AUSR0401,5814,POS,243982.0
3,2026-01-01,2601010547507676,P2P Credit,-82167.00,Успешные,6012,ePOS,3747452.0
4,2026-01-01,2601010547618620,Purchase,-101517.00,AUSR0401,5815,ePOS,820898.0



Total columns: 8


In [3]:
acquisition_file = DATA_DIR / "SAPP привлечения.csv"

df_acquisition = pd.read_csv(acquisition_file)

print(df_acquisition["secondary_category_filled"].value_counts(dropna=False))


secondary_category_filled
organic      415241
NaN          188264
Ticketon     146066
bank         137775
FTravel       99984
Arbuz         73851
FMobile       21645
FMedia        13218
Insurance     11261
FDrive        10864
DTP             929
Tours           380
Choco            22
Naimi            12
Name: count, dtype: int64


In [4]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect()

# Limit memory usage
con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")

audit_results = []

for file in csv_files:
    print(f"Processing: {file.name}")

    file_path = file.as_posix()

    query = """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customers,
            COUNT(*) - COUNT(customer_id) AS missing_customer_id
        FROM read_csv_auto(?, all_varchar=true)
    """

    result = con.execute(query, [file_path]).fetchone()

    audit_results.append({
        "file": file.name,
        "total_rows": result[0],
        "unique_customers": result[1],
        "missing_customer_id": result[2],
        "rows_minus_unique_customers": (
            result[0] - result[1]
        )
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)


Processing: SAPP Покупки у партнеров.csv
Processing: SAPP пользователи.csv
Processing: SAPP привлечения.csv
Processing: SAPP Процессы пользователей в приложении.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing: SAPP транзакции.csv


,file,total_rows,unique_customers,missing_customer_id,rows_minus_unique_customers
0,SAPP Покупки у партнеров.csv,215794,97622,0,118172
1,SAPP пользователи.csv,1114712,523340,0,591372
2,SAPP привлечения.csv,1119512,525207,113,594305
3,SAPP Процессы пользователей в приложении.csv,9499815,517321,0,8982494
4,SAPP транзакции.csv,6043146,233824,0,5809322


In [5]:
# Correct exact duplicate check

duplicate_results = []

for file in csv_files:
    print(f"Checking: {file.name}")

    file_path = file.as_posix()

    query = """
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) - COUNT(DISTINCT COLUMNS(*))
                AS duplicate_rows
        FROM read_csv_auto(
            ?,
            all_varchar=true
        )
    """

    # Count unique complete rows using SELECT DISTINCT
    query = """
        SELECT COUNT(*) AS unique_rows
        FROM (
            SELECT DISTINCT *
            FROM read_csv_auto(
                ?,
                all_varchar=true
            )
        )
    """

    unique_rows = con.execute(
        query, [file_path]
    ).fetchone()[0]

    total_rows = int(
        audit_df.loc[
            audit_df["file"] == file.name,
            "total_rows"
        ].iloc[0]
    )

    duplicate_results.append({
        "file": file.name,
        "total_rows": total_rows,
        "unique_rows": unique_rows,
        "exact_duplicates": total_rows - unique_rows,
        "duplicate_pct": round(
            (total_rows - unique_rows) / total_rows * 100,
            2
        )
    })

duplicates_df = pd.DataFrame(duplicate_results)

display(duplicates_df)


Checking: SAPP Покупки у партнеров.csv
Checking: SAPP пользователи.csv
Checking: SAPP привлечения.csv
Checking: SAPP Процессы пользователей в приложении.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking: SAPP транзакции.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,file,total_rows,unique_rows,exact_duplicates,duplicate_pct
0,SAPP Покупки у партнеров.csv,215794,215743,51,0.02
1,SAPP пользователи.csv,1114712,1113556,1156,0.10
2,SAPP привлечения.csv,1119512,950099,169413,15.13
3,SAPP Процессы пользователей в приложении.csv,9499815,9499782,33,0.00
4,SAPP транзакции.csv,6043146,6043146,0,0.00


In [6]:

# Check conflicting customer attributes

checks = {
    "SAPP пользователи.csv": [
        "city",
        "gender",
        "customer_age",
        "reg_date"
    ],
    "SAPP привлечения.csv": [
        "secondary_category_filled"
    ]
}

conflict_results = []

for filename, columns in checks.items():

    file_path = (DATA_DIR / filename).as_posix()

    print(f"\nChecking: {filename}")

    for column in columns:

        query = f"""
            SELECT COUNT(*)
            FROM (
                SELECT customer_id
                FROM read_csv_auto(
                    ?,
                    all_varchar=true
                )
                WHERE customer_id IS NOT NULL
                GROUP BY customer_id
                HAVING COUNT(DISTINCT "{column}") > 1
            )
        """

        conflicts = con.execute(
            query, [file_path]
        ).fetchone()[0]

        conflict_results.append({
            "file": filename,
            "column": column,
            "customers_with_conflicts": conflicts
        })

conflicts_df = pd.DataFrame(conflict_results)

display(conflicts_df)



Checking: SAPP пользователи.csv

Checking: SAPP привлечения.csv


,file,column,customers_with_conflicts
0,SAPP пользователи.csv,city,338626
1,SAPP пользователи.csv,gender,230330
2,SAPP пользователи.csv,customer_age,360557
3,SAPP пользователи.csv,reg_date,111178
4,SAPP привлечения.csv,secondary_category_filled,256944


In [7]:
# Analyze number of records per customer

files_to_check = [
    "SAPP пользователи.csv",
    "SAPP привлечения.csv"
]

for filename in files_to_check:

    print(f"\n{'=' * 60}")
    print(f"FILE: {filename}")
    print(f"{'=' * 60}")

    file_path = (DATA_DIR / filename).as_posix()

    query = """
        WITH customer_counts AS (
            SELECT
                customer_id,
                COUNT(*) AS records_per_customer
            FROM read_csv_auto(
                ?,
                all_varchar=true
            )
            WHERE customer_id IS NOT NULL
            GROUP BY customer_id
        )
        SELECT
            records_per_customer,
            COUNT(*) AS number_of_customers
        FROM customer_counts
        GROUP BY records_per_customer
        ORDER BY records_per_customer
    """

    result = con.execute(
        query, [file_path]
    ).df()

    display(result)



FILE: SAPP пользователи.csv


,records_per_customer,number_of_customers
0,1,159128
1,2,137052
2,3,227160



FILE: SAPP привлечения.csv


,records_per_customer,number_of_customers
0,1,159318
1,2,137586
2,3,228303


In [ ]:
# Analyze customer attribute consistency

filename = "SAPP пользователи.csv"
file_path = (DATA_DIR / filename).as_posix()

query = """
WITH customer_attributes AS (
    SELECT
        customer_id,
        COUNT(*) AS record_count,
        COUNT(DISTINCT city) AS unique_cities,
        COUNT(DISTINCT gender) AS unique_genders,
        COUNT(DISTINCT customer_age) AS unique_ages,
        COUNT(DISTINCT reg_date) AS unique_reg_dates
    FROM read_csv_auto(
        ?,
        all_varchar=true
    )
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)
SELECT
    record_count,
    COUNT(*) AS customers,
    SUM(CASE WHEN unique_cities > 1 THEN 1 ELSE 0 END)
        AS city_conflicts,
    SUM(CASE WHEN unique_genders > 1 THEN 1 ELSE 0 END)
        AS gender_conflicts,
    SUM(CASE WHEN unique_ages > 1 THEN 1 ELSE 0 END)
        AS age_conflicts,
    SUM(CASE WHEN unique_reg_dates > 1 THEN 1 ELSE 0 END)
        AS registration_conflicts
FROM customer_attributes
GROUP BY record_count
ORDER BY record_count
"""

consistency_df = con.execute(
    query, [file_path]
).df()

display(consistency_df)


,record_count,customers,city_conflicts,gender_conflicts,age_conflicts,registration_conflicts
0,1,159128,0.0,0.0,0.0,0.0
1,2,137052,118592.0,66099.0,133583.0,37489.0
2,3,227160,220034.0,164231.0,226974.0,73689.0


In [9]:
# Analyze registration date inconsistencies

file_path = (
    DATA_DIR / "SAPP пользователи.csv"
).as_posix()

query = """
WITH registration_dates AS (
    SELECT
        customer_id,
        TRY_CAST(reg_date AS DATE) AS registration_date
    FROM read_csv_auto(
        ?,
        all_varchar=true
    )
    WHERE customer_id IS NOT NULL
),

customer_date_ranges AS (
    SELECT
        customer_id,
        COUNT(DISTINCT registration_date)
            AS unique_registration_dates,
        MIN(registration_date) AS earliest_date,
        MAX(registration_date) AS latest_date,
        DATE_DIFF(
            'day',
            MIN(registration_date),
            MAX(registration_date)
        ) AS date_difference_days
    FROM registration_dates
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS total_customers,
    COUNT(*) FILTER (
        WHERE unique_registration_dates > 1
    ) AS customers_with_date_conflicts,
    ROUND(AVG(date_difference_days), 2)
        AS avg_date_difference,
    MAX(date_difference_days)
        AS max_date_difference,
    ROUND(
        MEDIAN(date_difference_days), 2
    ) AS median_date_difference
FROM customer_date_ranges
"""

registration_audit_df = con.execute(
    query, [file_path]
).df()

display(registration_audit_df)


,total_customers,customers_with_date_conflicts,avg_date_difference,max_date_difference,median_date_difference
0,523340,111178,4.93,130,0.0


In [10]:
# Analyze registration date conflict severity

file_path = (
    DATA_DIR / "SAPP пользователи.csv"
).as_posix()

query = """
WITH customer_dates AS (
    SELECT
        customer_id,
        MIN(TRY_CAST(reg_date AS DATE)) AS min_date,
        MAX(TRY_CAST(reg_date AS DATE)) AS max_date,
        COUNT(DISTINCT TRY_CAST(reg_date AS DATE))
            AS unique_dates
    FROM read_csv_auto(
        ?,
        all_varchar=true
    )
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
),

date_conflicts AS (
    SELECT
        customer_id,
        DATE_DIFF(
            'day',
            min_date,
            max_date
        ) AS difference_days
    FROM customer_dates
    WHERE unique_dates > 1
)

SELECT
    CASE
        WHEN difference_days <= 7 THEN '01. 1-7 days'
        WHEN difference_days <= 30 THEN '02. 8-30 days'
        WHEN difference_days <= 60 THEN '03. 31-60 days'
        WHEN difference_days <= 90 THEN '04. 61-90 days'
        ELSE '05. 91+ days'
    END AS difference_group,

    COUNT(*) AS customers,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage

FROM date_conflicts
GROUP BY difference_group
ORDER BY difference_group
"""

date_conflict_distribution = con.execute(
    query, [file_path]
).df()

display(date_conflict_distribution)


,difference_group,customers,percentage
0,01. 1-7 days,49734,44.73
1,02. 8-30 days,28934,26.02
2,03. 31-60 days,18797,16.91
3,04. 61-90 days,9841,8.85
4,05. 91+ days,3872,3.48


In [11]:
# Analyze missing values in all datasets

missing_results = []

for file in csv_files:
    print(f"Checking missing values: {file.name}")

    file_path = file.as_posix()

    # Read column names without loading the full dataset
    columns = con.execute(
        """
        DESCRIBE SELECT *
        FROM read_csv_auto(?, all_varchar=true)
        """,
        [file_path]
    ).df()["column_name"].tolist()

    # Count NULL and empty string values
    expressions = []

    for column in columns:
        safe_column = column.replace('"', '""')

        expressions.append(
            f"""
            SUM(
                CASE
                    WHEN "{safe_column}" IS NULL
                      OR TRIM("{safe_column}") = ''
                    THEN 1 ELSE 0
                END
            ) AS "{safe_column}"
            """
        )

    query = f"""
        SELECT
            COUNT(*) AS total_rows,
            {", ".join(expressions)}
        FROM read_csv_auto(
            ?,
            all_varchar=true
        )
    """

    result = con.execute(query, [file_path]).df()

    total_rows = int(result["total_rows"].iloc[0])

    for column in columns:
        missing_count = int(result[column].iloc[0])

        missing_results.append({
            "file": file.name,
            "column": column,
            "missing_count": missing_count,
            "missing_pct": round(
                missing_count / total_rows * 100,
                2
            )
        })

missing_df = pd.DataFrame(missing_results)

display(
    missing_df.sort_values(
        "missing_pct",
        ascending=False
    )
)


Checking missing values: SAPP Покупки у партнеров.csv
Checking missing values: SAPP пользователи.csv
Checking missing values: SAPP привлечения.csv
Checking missing values: SAPP Процессы пользователей в приложении.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking missing values: SAPP транзакции.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,file,column,missing_count,missing_pct
12,SAPP привлечения.csv,secondary_category_filled,183744,16.41
26,SAPP транзакции.csv,terminal_type,634432,10.50
23,SAPP транзакции.csv,transaction_sum,577830,9.56
19,SAPP Процессы пользователей в приложении.csv,completed_at,327690,3.45
16,SAPP Процессы пользователей в приложении.csv,lang,13637,0.14
13,SAPP привлечения.csv,customer_id,113,0.01
0,SAPP Покупки у партнеров.csv,purchase_date,0,0.00
15,SAPP Процессы пользователей в приложении.csv,process_code,0,0.00
25,SAPP транзакции.csv,mcc,0,0.00
24,SAPP транзакции.csv,transaction_status,0,0.00


In [12]:
# Investigate missing values by transaction and process status

transactions_path = (
    DATA_DIR / "SAPP транзакции.csv"
).as_posix()

processes_path = (
    DATA_DIR / "SAPP Процессы пользователей в приложении.csv"
).as_posix()

# 1. Transaction amount missing by operation and status
transaction_query = """
SELECT
    operation_type,
    transaction_status,
    COUNT(*) AS total_transactions,

    COUNT(*) FILTER (
        WHERE transaction_sum IS NULL
           OR TRIM(transaction_sum) = ''
    ) AS missing_amount,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE transaction_sum IS NULL
               OR TRIM(transaction_sum) = ''
        ) / COUNT(*),
        2
    ) AS missing_pct

FROM read_csv_auto(
    ?,
    all_varchar=true
)
GROUP BY operation_type, transaction_status
ORDER BY total_transactions DESC
"""

transaction_status_df = con.execute(
    transaction_query,
    [transactions_path]
).df()

print("TRANSACTION STATUS ANALYSIS")
display(transaction_status_df.head(30))


# 2. Missing completion time by process status
process_query = """
SELECT
    status,
    COUNT(*) AS total_processes,

    COUNT(*) FILTER (
        WHERE completed_at IS NULL
           OR TRIM(completed_at) = ''
    ) AS missing_completed_at,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE completed_at IS NULL
               OR TRIM(completed_at) = ''
        ) / COUNT(*),
        2
    ) AS missing_pct

FROM read_csv_auto(
    ?,
    all_varchar=true
)
GROUP BY status
ORDER BY total_processes DESC
"""

process_status_df = con.execute(
    process_query,
    [processes_path]
).df()

print("\nPROCESS STATUS ANALYSIS")
display(process_status_df)


TRANSACTION STATUS ANALYSIS


,operation_type,transaction_status,total_transactions,missing_amount,missing_pct
0,Purchase,AUSR0401,3278597,0,0.00
1,P2P Debit,Успешные,919180,1457,0.16
2,P2P Credit,Успешные,858686,0,0.00
3,Purchase,RESP7150,391682,391682,100.00
4,Transaction Cost Inquiry,Успешные,221704,0,0.00
5,Purchase,Успешные,99356,0,0.00
6,Purchase,RESP7152,54189,54189,100.00
7,Purchase,RESP0002,40674,40674,100.00
8,ATM Cash withdrawal,AUSR0401,20003,0,0.00
9,Payment transaction,Успешные,19662,0,0.00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PROCESS STATUS ANALYSIS


,status,total_processes,missing_completed_at,missing_pct
0,COMPLETED,8290168,25,0.00
1,DECLINED,559456,0,0.00
2,STARTED,326609,326609,100.00
3,ERROR,303193,9,0.00
4,TIMEOUT,20318,1007,4.96
5,REVIEW,40,40,100.00
6,PENDING,29,0,0.00
7,ON_OPENING,1,0,0.00
8,OPENED_PARTIALLY,1,0,0.00


In [13]:
# Numeric data validation

numeric_checks = {
    "SAPP пользователи.csv": ["customer_age"],
    "SAPP транзакции.csv": ["transaction_sum"],
    "SAPP Покупки у партнеров.csv": [
        "purchase_amount",
        "cashback_amount",
        "counter"
    ]
}

numeric_results = []

for filename, columns in numeric_checks.items():

    file_path = (DATA_DIR / filename).as_posix()

    print(f"Checking: {filename}")

    for column in columns:
        safe_column = column.replace('"', '""')

        query = f"""
        WITH numeric_data AS (
            SELECT
                TRY_CAST(
                    "{safe_column}" AS DOUBLE
                ) AS value,
                "{safe_column}" AS raw_value
            FROM read_csv_auto(
                ?,
                all_varchar=true
            )
        )
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE value IS NULL
                  AND raw_value IS NOT NULL
                  AND TRIM(raw_value) <> ''
            ) AS invalid_numeric,
            COUNT(*) FILTER (
                WHERE value < 0
            ) AS negative_count,
            COUNT(*) FILTER (
                WHERE value = 0
            ) AS zero_count,
            MIN(value) AS min_value,
            MAX(value) AS max_value,
            ROUND(AVG(value), 2) AS mean_value,
            MEDIAN(value) AS median_value
        FROM numeric_data
        """

        result = con.execute(
            query, [file_path]
        ).df().iloc[0]

        numeric_results.append({
            "file": filename,
            "column": column,
            **result.to_dict()
        })

numeric_audit_df = pd.DataFrame(numeric_results)

display(numeric_audit_df)


Checking: SAPP пользователи.csv
Checking: SAPP транзакции.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Checking: SAPP Покупки у партнеров.csv


,file,column,total_rows,invalid_numeric,negative_count,zero_count,min_value,max_value,mean_value,median_value
0,SAPP пользователи.csv,customer_age,1114712.0,0.0,0.0,0.0,5.0,9.600000e+01,29.48,29.00
1,SAPP транзакции.csv,transaction_sum,6043146.0,0.0,5184004.0,0.0,-101667.0,6.659948e+09,-68602.44,-97834.50
2,SAPP Покупки у партнеров.csv,purchase_amount,215794.0,0.0,183968.0,1.0,-101667.0,9.821571e+06,-34053.74,-80667.00
3,SAPP Покупки у партнеров.csv,cashback_amount,215794.0,0.0,215655.0,0.0,-101667.0,2.851350e+05,-95487.17,-98448.18
4,SAPP Покупки у партнеров.csv,counter,215794.0,0.0,0.0,0.0,1.0,4.700000e+01,1.16,1.00


In [14]:
# Explore available products for cross-sell prediction

partners_path = (
    DATA_DIR / "SAPP Покупки у партнеров.csv"
).as_posix()

query = """
SELECT
    app_name,
    COUNT(*) AS total_purchases,
    COUNT(DISTINCT customer_id) AS unique_customers,
    MIN(TRY_CAST(purchase_date AS DATE)) AS first_purchase,
    MAX(TRY_CAST(purchase_date AS DATE)) AS last_purchase
FROM read_csv_auto(
    ?,
    all_varchar=true
)
GROUP BY app_name
ORDER BY total_purchases DESC
"""

product_distribution_df = con.execute(
    query,
    [partners_path]
).df()

display(product_distribution_df)


,app_name,total_purchases,unique_customers,first_purchase,last_purchase
0,flights,48509,29493,2024-07-21,2026-05-12
1,groceries,45392,13582,2024-11-29,2026-05-12
2,Тикетон,45273,32137,2024-12-07,2026-04-06
3,train_tickets,35580,20542,2025-06-25,2026-05-12
4,ticketon,22921,18212,2026-04-06,2026-05-12
5,fmedia,12429,10241,2025-12-02,2026-05-12
6,telecom,2983,2158,2026-01-01,2026-05-12
7,ftelecom,972,969,2026-04-28,2026-05-12
8,Freedom Mobile,610,592,2025-11-19,2026-04-06
9,drive,394,379,2025-11-08,2026-05-12
